In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviews.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

In [3]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_id').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0).filter( col('score') <= 10)

In [4]:
from recommenders import train_item_item

trained_ii_model, score_history = train_item_item(anime_data)

In [6]:
from recommenders import get_similar_items_for_user
target_uid = final_data.select('user_id').first()[0]
get_similar_items_for_user(target_uid, final_data, trained_ii_model, score_history, 100)

Get history: 0.9906985759735107 secs!
Size of history: 109
Get candidates: 0.3775045871734619 secs!
Process results: 0.0049228668212890625 secs!
Total time: 1.3733110427856445 secs!


,anime_id,avg_sim,final_score
548,10943,0.820049,1.000000
547,5992,0.820049,1.000000
545,10460,0.820049,1.000000
544,11739,0.820049,1.000000
1416,23447,0.818919,0.998527
...,...,...,...
2915,260,0.735340,0.889625
2925,394,0.735340,0.889625
2904,836,0.735340,0.889625
2394,33581,0.733599,0.887356


In [7]:
def ii_model(user_id, n=10):
    return get_similar_items_for_user(user_id, final_data, trained_ii_model, score_history, n)

In [8]:
from recommenders import hybridV1
from pyspark.ml.recommendation import ALSModel
ui_model = ALSModel.load('/home/jovyan/work/models/user-item')

hybridV1(target_uid, ui_model, ii_model, final_data, score_history, spark, 10)

Get history: 0.5501480102539062 secs!
Size of history: 109
Get candidates: 0.1471853256225586 secs!
Process results: 0.0012888908386230469 secs!
Total time: 0.698714017868042 secs!
+---------+------------------+-------------------+-------------------+
|anime_id |ui_score          |ii_score           |hybrid_score       |
+---------+------------------+-------------------+-------------------+
|36885.0  |14.139758110046387|0.5506317019462585 |0.6845119177462428 |
|37403.0  |13.240625381469727|0.5299467444419861 |0.646603384391798  |
|34928.0  |12.750710487365723|0.43475112318992615|0.5839855213226623 |
|31165.0  |13.180425643920898|0.3926224112510681 |0.5760955845275678 |
|35267.0  |11.817183494567871|0.4409980773925781 |0.5584884675131436 |
|39178.0  |12.380205154418945|0.3866478204727173 |0.5485747347429613 |
|3568.0   |12.830952644348145|0.33271217346191406|0.5354261500581516 |
|3.859215 |0.8057902119534109|0.9999999985976952 |0.5003967027239414 |
|8.023775 |0.8057902119534109|0.999999

DataFrame[anime_id: float, ui_score: double, ii_score: double, ui_score_norm: double, ii_score_norm: double, hybrid_score: double]